In [ ]:
import nltk
import numpy as np
import networkx as nx
import re
import pdfplumber

from nltk.tokenize import sent_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


# Download required nltk data
nltk.download('punkt')


# ---------------------------------
# READ DOCUMENT (TXT or PDF)
# ---------------------------------

def read_document(file_path):

    if file_path.lower().endswith(".txt"):
        with open(file_path, "r", encoding="utf8") as f:
            text = f.read()

    elif file_path.lower().endswith(".pdf"):
        text = ""
        with pdfplumber.open(file_path) as pdf:
            for page in pdf.pages:
                extracted = page.extract_text()
                if extracted:
                    text += extracted

    else:
        raise ValueError("Unsupported file format. Use .txt or .pdf")

    return text


# ---------------------------------
# PREPROCESS TEXT
# ---------------------------------

def preprocess_text(text):

    text = re.sub(r'\s+', ' ', text)

    sentences = sent_tokenize(text)

    return sentences


# ---------------------------------
# SECTION DETECTION (NOVELTY)
# ---------------------------------

def detect_sections(text):

    sections = {
        "FACTS": "",
        "ARGUMENTS": "",
        "JUDGMENT": ""
    }

    current_section = None

    lines = text.split("\n")

    for line in lines:

        l = line.lower()

        if "facts" in l:
            current_section = "FACTS"

        elif "argument" in l:
            current_section = "ARGUMENTS"

        elif "judgment" in l or "decision" in l or "order" in l:
            current_section = "JUDGMENT"

        if current_section:
            sections[current_section] += " " + line

    return sections


# ---------------------------------
# SUMMARIZATION FUNCTION
# ---------------------------------

def summarize_text(text, top_n=3):

    sentences = preprocess_text(text)

    if len(sentences) == 0:
        return "No content available."

    if len(sentences) <= top_n:
        return " ".join(sentences)

    vectorizer = TfidfVectorizer(stop_words="english")

    tfidf_matrix = vectorizer.fit_transform(sentences)

    similarity_matrix = cosine_similarity(tfidf_matrix, tfidf_matrix)

    np.fill_diagonal(similarity_matrix, 0)

    graph = nx.from_numpy_array(similarity_matrix)

    scores = nx.pagerank(graph)

    ranked_sentences = sorted(
        ((scores[i], s) for i, s in enumerate(sentences)),
        reverse=True
    )

    summary = " ".join([ranked_sentences[i][1] for i in range(top_n)])

    return summary


# ---------------------------------
# QUERY BASED SEARCH (NOVELTY)
# ---------------------------------

def query_search(query, text):

    sentences = preprocess_text(text)

    vectorizer = TfidfVectorizer(stop_words="english")

    vectors = vectorizer.fit_transform(sentences + [query])

    similarity = cosine_similarity(vectors[-1], vectors[:-1])

    best_index = np.argmax(similarity)

    return sentences[best_index]


# ---------------------------------
# MAIN PROGRAM
# ---------------------------------

def main():

    print("\nINTELLIGENT LEGAL DOCUMENT SUMMARIZATION SYSTEM\n")

    file_path = input("Enter document path (.txt or .pdf): ")

    try:
        text = read_document(file_path)
    except Exception as e:
        print("Error:", e)
        return

    sections = detect_sections(text)

    print("\n------------- SECTION SUMMARIES -------------\n")

    found = False

    for section, content in sections.items():

        if content.strip() != "":
            found = True

            print(f"\n{section} SUMMARY:\n")

            summary = summarize_text(content)

            print(summary)

    if not found:
        print("No structured sections detected. Generating overall summary...\n")
        print(summarize_text(text))

    # ---------------------------------
    # CONTINUOUS QUERY SYSTEM
    # ---------------------------------

    print("\n------------- QUERY SEARCH -------------\n")
    print("Ask questions about the document.")
    print("Type 'quit' to exit.\n")

    while True:

        query = input("Enter your question: ")

        if query.lower() == "quit" or query.lower() == "exit":
            print("\nExiting query system. Thank you!\n")
            break

        result = query_search(query, text)

        print("\nMost Relevant Sentence:\n")
        print(result)
        print("\n---------------------------------------\n")


# ---------------------------------
# RUN PROGRAM
# ---------------------------------

if __name__ == "__main__":
    main()